## Packages

In [3]:
import sys
from pathlib import Path

import spatialdata as spd
import matplotlib.pyplot as plt
import anndata as ad
import squidpy as sq
import numpy as np

# --- Make project-shared utilities importable across machines/envs ---
# Jupyter's CWD is wherever the kernel started (often $HOME), not where the
# notebook file lives. So we try, in order:
#   1. VS Code's injected notebook path (__vsc_ipynb_file__)
#   2. Current working directory
#   3. Known code roots (laptop + server)
# For each, we walk up until we find a dir containing misc/plot_style.py.
_candidates = []
_nb = globals().get("__vsc_ipynb_file__")
if _nb:
    _candidates.append(Path(_nb).resolve().parent)
_candidates.append(Path.cwd().resolve())
_candidates.extend([
    Path("/home/janzules/spatial/CAR-T/code"),
    Path("/Users/janzules/Roselab/Spatial/CAR_T/code"),
])

_added = False
for _start in _candidates:
    for _parent in [_start, *_start.parents]:
        if (_parent / "misc" / "plot_style.py").exists():
            if str(_parent) not in sys.path:
                sys.path.insert(0, str(_parent))
            _added = True
            break
    if _added:
        break

if not _added:
    raise RuntimeError(
        "Could not locate code/misc/plot_style.py from any known root. "
        "Add the code/ directory to sys.path manually."
    )

from misc.plot_style import CELL_TYPE_COLORS, get_cell_colors

RuntimeError: Could not locate code/misc/plot_style.py from any known root. Add the code/ directory to sys.path manually.

In [2]:
from pathlib import Path

# Data

In [3]:
# proj_folder   = Path("/coh_labs/yunroseli/Jona/CAR-T/") # Most up to date files
# zarr_file     = proj_folder / "data/zarr/CellCharterClusters_c2l_annotated"
zarr_file     = "/coh_labs/yunroseli/Jona/CAR-T/data/zarr/fullDataset/processing_Zarr/1_C2l_annotations_400"
# code_folder   = Path("/home/janzules/spatial/CAR-T/code")
# hallmark_file = code_folder / "references/Mouse_Hallmark.gmt"

## Loading Data

In [4]:
#Reading in files

sdata = spd.read_zarr(zarr_file)
adata = sdata.tables['segmentation_counts']

In [9]:
# Dropping unknown cells
adata = adata[
    (adata.obs['c2l_consolidated'] != 'Unknown') & 
    (adata.obs['c2l_consolidated'] != 'Erythrocyte') &
    (adata.obs['c2l_consolidated'] != 'B')
].copy()


In [14]:
adata.obs['c2l_consolidated'].unique()

array(['Classical_Mono', 'Neutrophil', 'N1_like_Neu', 'Nonclassical_Mono',
       'M1_like_Mac', 'CD8_T', 'M2_like_Mac', 'Fibroblast', 'NK',
       'Cancer_cell', 'cDC', 'Endothelial', 'N2_like_Neu', 'pDC', 'CD4_T',
       'Monocyte', 'Treg', 'NKT', 'Intermediate_Mac', 'Macrophage',
       'Tcell', 'DC'], dtype=object)

# Visualizing

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# ----------------------------
# User settings
# ----------------------------
cell_col = "c2l_consolidated"
sample_col = "mouse"       # most specific tissue/sample label
condition_col = "treatment"  # or "condition", depending on your obs columns

# ----------------------------
# Basic checks
# ----------------------------
required_cols = [cell_col, sample_col, condition_col]
missing = [c for c in required_cols if c not in adata.obs.columns]

if missing:
    raise ValueError(f"Missing columns in adata.obs: {missing}")

obs = adata.obs[required_cols].copy()
obs = obs.dropna(subset=[cell_col, sample_col, condition_col])

obs[cell_col] = obs[cell_col].astype(str)
obs[sample_col] = obs[sample_col].astype(str)
obs[condition_col] = obs[condition_col].astype(str)

# ----------------------------
# Counts and fractions per sample
# ----------------------------
sample_counts = pd.crosstab(obs[sample_col], obs[cell_col])
sample_fractions = sample_counts.div(sample_counts.sum(axis=1), axis=0)

sample_metadata = (
    obs[[sample_col, condition_col]]
    .drop_duplicates()
    .set_index(sample_col)
)

sample_fractions = sample_fractions.join(sample_metadata)

print("Sample-level count table:")
display(sample_counts.head())

print("Sample-level fraction table:")
display(sample_fractions.head())

print("\nNumber of cells per sample:")
display(sample_counts.sum(axis=1).sort_values(ascending=False))

Sample-level count table:


c2l_consolidated,CD4_T,CD8_T,Cancer_cell,Classical_Mono,DC,Endothelial,Fibroblast,Intermediate_Mac,M1_like_Mac,M2_like_Mac,...,N1_like_Neu,N2_like_Neu,NK,NKT,Neutrophil,Nonclassical_Mono,Tcell,Treg,cDC,pDC
mouse,,,,,,,,,,,,,,,,,,,,,
CyPSCA_1_1,1198,5347,9022,4474,1,282,198,106,1356,452,...,5643,1225,2400,917,9243,337,19,384,532,545
CyPSCA_1_2,417,1483,12378,6213,0,479,199,57,461,630,...,6608,2658,722,247,2893,190,6,275,174,112
CyPSCA_1_3,19,248,634,3527,0,243,179,5,1891,1844,...,5468,1000,247,29,2418,109,0,39,103,32
CyPSCA_1_4,97,847,1276,790,0,244,121,33,118,1186,...,2584,1033,252,135,2160,79,2,101,60,99
CyPSCA_2_1,37,435,375,11353,4,183,738,12,2771,1351,...,9403,1025,266,93,11864,370,1,114,507,238


Sample-level fraction table:


,CD4_T,CD8_T,Cancer_cell,Classical_Mono,DC,Endothelial,Fibroblast,Intermediate_Mac,M1_like_Mac,M2_like_Mac,...,N2_like_Neu,NK,NKT,Neutrophil,Nonclassical_Mono,Tcell,Treg,cDC,pDC,treatment
mouse,,,,,,,,,,,,,,,,,,,,,
CyPSCA_1_1,0.027417,0.122371,0.206477,0.102392,0.000023,0.006454,0.004531,0.002426,0.031033,0.010344,...,0.028035,0.054926,0.020986,0.211535,0.007713,0.000435,0.008788,0.012175,0.012473,CyPSCA_Tu1
CyPSCA_1_2,0.011516,0.040953,0.341820,0.171573,0.000000,0.013228,0.005495,0.001574,0.012731,0.017398,...,0.073401,0.019938,0.006821,0.079891,0.005247,0.000166,0.007594,0.004805,0.003093,CyPSCA_Tu1
CyPSCA_1_3,0.001053,0.013743,0.035132,0.195445,0.000000,0.013466,0.009919,0.000277,0.104788,0.102183,...,0.055414,0.013687,0.001607,0.133991,0.006040,0.000000,0.002161,0.005708,0.001773,CyPSCA_Tu1
CyPSCA_1_4,0.008644,0.075477,0.113705,0.070397,0.000000,0.021743,0.010782,0.002941,0.010515,0.105685,...,0.092051,0.022456,0.012030,0.192479,0.007040,0.000178,0.009000,0.005347,0.008822,CyPSCA_Tu1
CyPSCA_2_1,0.000899,0.010568,0.009111,0.275819,0.000097,0.004446,0.017930,0.000292,0.067321,0.032822,...,0.024902,0.006462,0.002259,0.288234,0.008989,0.000024,0.002770,0.012317,0.005782,CyPSCA_Tu2



Number of cells per sample:


mouse
NoTx_2_2          80627
NoTx_2_3          77004
CyT72_2_4         74913
NoTX_2_4          64319
RTCyPSCA_2_3      58795
NoTx_1_3          50862
NoTx_2_1          50403
RTCyT72_2_4       47319
CyPSCA_1_1        43695
CyPSCA_2_1        41161
RTCyPSCA_2_4      39811
RTCyPSCA_1_3      37228
CyT72_2_3         36918
CyPSCA_1_2        36212
RTCyT72_1_2       35948
CyPSCA_2_4        31477
RTCyPSCA_1_4      29824
NoTx_1_4          29674
CyPSCA_2_2        29120
RTCyT72_2_2       28881
RTCyT72_2_1       28858
RTCyT72_1_4       28238
CyPSCA_2_3        27592
NoTx_1_1          26589
RTCyPSCA_1_2_2    22702
CyT72_1_4         19759
NoTx_1_2          18238
CyPSCA_1_3        18046
RTCyT72_1_1       14733
CyT72_1_2         14372
RTCyPSCA_1_2_1    12142
CyPSCA_1_4        11222
dtype: int64

In [ ]:
# Separate numeric cell-type columns from metadata
cell_type_cols = sample_counts.columns.tolist()

# Optional: sort samples by condition, then sample name
plot_df = sample_fractions.copy()
plot_df = plot_df.sort_values([condition_col, sample_col])

plot_cell_frac = plot_df[cell_type_cols]

fig, ax = plt.subplots(figsize=(14, 6))

bottom = np.zeros(plot_cell_frac.shape[0])

for ct in cell_type_cols:
    ax.bar(
        plot_cell_frac.index,
        plot_cell_frac[ct],
        bottom=bottom,
        label=ct,
        color=CELL_TYPE_COLORS.get(ct, "#999999"),
    )
    bottom += plot_cell_frac[ct].values

ax.set_ylabel("Fraction of all cells")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("Sample")
ax.set_title("Cellular composition per sample")
ax.tick_params(axis="x", rotation=90)

ax.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
    title="Cell type"
)

plt.tight_layout()
plt.show()

print("Summary:")
print("Each bar is one tissue/sample. Fractions are normalized within sample, so all bars sum to 100%.")

In [ ]:
# Define rare cell types based on global abundance
# Rare cell types lumped to 1%
min_global_fraction = 0.01  # 1%

global_fraction = obs[cell_col].value_counts(normalize=True)
major_cell_types = global_fraction[global_fraction >= min_global_fraction].index.tolist()
rare_cell_types = global_fraction[global_fraction < min_global_fraction].index.tolist()

print("Major cell types:")
print(major_cell_types)

print("\nCollapsed into Other:")
print(rare_cell_types)

obs_collapsed = obs.copy()
obs_collapsed["cell_type_collapsed"] = np.where(
    obs_collapsed[cell_col].isin(major_cell_types),
    obs_collapsed[cell_col],
    "Other"
)

collapsed_counts = pd.crosstab(obs_collapsed[sample_col], obs_collapsed["cell_type_collapsed"])
collapsed_fractions = collapsed_counts.div(collapsed_counts.sum(axis=1), axis=0)

collapsed_fractions = collapsed_fractions.join(sample_metadata)
collapsed_fractions = collapsed_fractions.sort_values([condition_col, sample_col])

collapsed_cell_cols = collapsed_counts.columns.tolist()
plot_df = collapsed_fractions[collapsed_cell_cols]

fig, ax = plt.subplots(figsize=(12, 5))

bottom = np.zeros(plot_df.shape[0])

for ct in collapsed_cell_cols:
    ax.bar(
        plot_df.index,
        plot_df[ct],
        bottom=bottom,
        label=ct,
        color=CELL_TYPE_COLORS.get(ct, "#999999"),
    )
    bottom += plot_df[ct].values

ax.set_ylabel("Fraction of all cells")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("Sample")
ax.set_title("Cellular composition per sample, rare cell types collapsed")
ax.tick_params(axis="x", rotation=90)

ax.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
    title="Cell type"
)

plt.tight_layout()
plt.show()

print("Summary:")
print(f"Cell types below {min_global_fraction:.0%} global abundance were collapsed into Other.")

# Lineage-level composition

In [ ]:
lineage_map = {
    "Cancer_cell": "Tumor",

    "N1_like_Neu": "Neutrophil",
    "N2_like_Neu": "Neutrophil",

    "M1_like_Mac": "Macrophage/Monocyte",
    "M2_like_Mac": "Macrophage/Monocyte",
    "Intermediate_Mac": "Macrophage/Monocyte",
    "Classical_Mono": "Macrophage/Monocyte",
    "Nonclassical_Mono": "Macrophage/Monocyte",

    "CD8_T": "Lymphoid",
    "CD4_T": "Lymphoid",
    "Treg": "Lymphoid",
    "NK": "Lymphoid",
    "NKT": "Lymphoid",
    "B": "Lymphoid",

    "cDC": "Dendritic cell",
    "pDC": "Dendritic cell",

    "Fibroblast": "Stromal",
    "Endothelial": "Stromal",

    "Erythrocyte": "Erythrocyte"
}

obs_lineage = obs.copy()
obs_lineage["lineage"] = obs_lineage[cell_col].map(lineage_map).fillna("Other")

lineage_counts = pd.crosstab(obs_lineage[sample_col], obs_lineage["lineage"])
lineage_fractions = lineage_counts.div(lineage_counts.sum(axis=1), axis=0)

lineage_fractions = lineage_fractions.join(sample_metadata)
lineage_fractions = lineage_fractions.sort_values([condition_col, sample_col])

lineage_cols = lineage_counts.columns.tolist()
plot_df = lineage_fractions[lineage_cols]

fig, ax = plt.subplots(figsize=(12, 5))

bottom = np.zeros(plot_df.shape[0])

for lineage in lineage_cols:
    ax.bar(
        plot_df.index,
        plot_df[lineage],
        bottom=bottom,
        label=lineage
    )
    bottom += plot_df[lineage].values

ax.set_ylabel("Fraction of all cells")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("Sample")
ax.set_title("Lineage-level cellular composition per sample")
ax.tick_params(axis="x", rotation=90)

ax.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
    title="Lineage"
)

plt.tight_layout()
plt.show()

print("Summary:")
print("This plot collapses detailed cell annotations into broader biological compartments.")

In [ ]:
# Statistical measure per group condition

condition_summary = (
    sample_fractions
    .groupby(condition_col)[cell_type_cols]
    .agg(["mean", "std", "sem"])
)

display(condition_summary)

condition_mean = sample_fractions.groupby(condition_col)[cell_type_cols].mean()

fig, ax = plt.subplots(figsize=(10, 5))

bottom = np.zeros(condition_mean.shape[0])

for ct in cell_type_cols:
    ax.bar(
        condition_mean.index,
        condition_mean[ct],
        bottom=bottom,
        label=ct,
        color=CELL_TYPE_COLORS.get(ct, "#999999"),
    )
    bottom += condition_mean[ct].values

ax.set_ylabel("Mean fraction of all cells")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("Treatment group")
ax.set_title("Mean cellular composition by treatment group")
ax.tick_params(axis="x", rotation=45)

ax.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
    title="Cell type"
)

plt.tight_layout()
plt.show()

print("Summary:")
print("Treatment-group bars are calculated from per-sample fractions, not pooled cells.")

In [ ]:
cell_of_interest = "CD8_T"

focus_df = sample_fractions[[cell_of_interest, condition_col]].copy()
focus_df = focus_df.rename(columns={cell_of_interest: "fraction"})
focus_df = focus_df.sort_values(condition_col)

fig, ax = plt.subplots(figsize=(7, 5))

conditions = focus_df[condition_col].unique()
x_positions = np.arange(len(conditions))

for i, cond in enumerate(conditions):
    y = focus_df.loc[focus_df[condition_col] == cond, "fraction"]

    # sample dots
    ax.scatter(
        np.repeat(i, len(y)),
        y,
        alpha=0.8
    )

    # condition mean
    ax.hlines(
        y.mean(),
        i - 0.25,
        i + 0.25,
        linewidth=3
    )

ax.set_xticks(x_positions)
ax.set_xticklabels(conditions, rotation=45, ha="right")
ax.set_ylabel(f"{cell_of_interest} / all cells")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title(f"{cell_of_interest} abundance by treatment group")

plt.tight_layout()
plt.show()

print("Summary:")
print(f"This plot shows {cell_of_interest} as a fraction of all segmented cells in each sample.")

In [ ]:
immune_cell_types = [
    "N2_like_Neu",
    "M2_like_Mac",
    "Classical_Mono",
    "N1_like_Neu",
    "NK",
    "CD8_T",
    "M1_like_Mac",
    "B",
    "cDC",
    "CD4_T",
    "Intermediate_Mac",
    "Nonclassical_Mono",
    "pDC",
    "NKT",
    "Treg"
]

cell_of_interest = "CD8_T"

immune_obs = obs[obs[cell_col].isin(immune_cell_types)].copy()

immune_counts = pd.crosstab(immune_obs[sample_col], immune_obs[cell_col])
immune_fractions = immune_counts.div(immune_counts.sum(axis=1), axis=0)

immune_fractions = immune_fractions.join(sample_metadata)

focus_df = immune_fractions[[cell_of_interest, condition_col]].copy()
focus_df = focus_df.rename(columns={cell_of_interest: "fraction"})
focus_df = focus_df.dropna()
focus_df = focus_df.sort_values(condition_col)

fig, ax = plt.subplots(figsize=(7, 5))

conditions = focus_df[condition_col].unique()
x_positions = np.arange(len(conditions))

for i, cond in enumerate(conditions):
    y = focus_df.loc[focus_df[condition_col] == cond, "fraction"]

    ax.scatter(
        np.repeat(i, len(y)),
        y,
        alpha=0.8
    )

    ax.hlines(
        y.mean(),
        i - 0.25,
        i + 0.25,
        linewidth=3
    )

ax.set_xticks(x_positions)
ax.set_xticklabels(conditions, rotation=45, ha="right")
ax.set_ylabel(f"{cell_of_interest} / immune cells")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title(f"{cell_of_interest} fraction among immune cells")

plt.tight_layout()
plt.show()

print("Summary:")
print(f"This plot shows {cell_of_interest} as a fraction of immune cells only.")